# Spark Transformation Prototype
This notebook demonstrates a basic PySpark pipeline: Ingestion, Transformation, and Export.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("DataEng_ML_Spark_Prototype") \
    .getOrCreate()  # gets existing session or creates new one

print("Spark Session Created")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/03 17:53:45 WARN Utils: Your hostname, LAPTOP-HP, resolves to a loopback address: 127.0.1.1; using 192.168.1.110 instead (on interface wlo1)
26/06/03 17:53:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/03 17:53:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session Created


In [2]:
# 2. Ingestion: Read raw data
raw_data_path = "data/raw/sample.csv"
df = spark.read.csv(raw_data_path, header=True, inferSchema=True)

print(f"Read {df.count()} rows")
display(df.show(5))
display(list(df.schema))

Read 1244245 rows
+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code|   brand|  price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|2019-10-01 00:00:00|      view|  44600062|2103807459595387724|                NULL|shiseido|  35.79|541312140|72d76fde-8bb3-4e0...|
|2019-10-01 00:00:00|      view|   3900821|2053013552326770905|appliances.enviro...|    aqua|   33.2|554748717|9333dfbd-b87a-470...|
|2019-10-01 00:00:01|      view|  17200506|2053013559792632471|furniture.living_...|    NULL|  543.1|519107250|566511c2-e2e3-422...|
|2019-10-01 00:00:01|      view|   1307067|2053013558920217191|  computers.notebook|  lenovo| 251.74|550050854|7c90fc70-0e80-459...|
|2019-10-01 00:00:04|      view|   1004237|20530135

None

[StructField('event_time', TimestampType(), True),
 StructField('event_type', StringType(), True),
 StructField('product_id', IntegerType(), True),
 StructField('category_id', LongType(), True),
 StructField('category_code', StringType(), True),
 StructField('brand', StringType(), True),
 StructField('price', DoubleType(), True),
 StructField('user_id', IntegerType(), True),
 StructField('user_session', StringType(), True)]

In [3]:
# 3. Transformation: Filter for 'view' events and select columns
transformed_df = df.filter(col("event_type") == "view") \
    .select("event_time", "brand", "price", "user_id") \
    .limit(1000) # Limit for prototype validation

print("Transformation Complete")
display(transformed_df.show(5))
display(list(transformed_df.schema))

Transformation Complete
+-------------------+--------+-------+---------+
|         event_time|   brand|  price|  user_id|
+-------------------+--------+-------+---------+
|2019-10-01 00:00:00|shiseido|  35.79|541312140|
|2019-10-01 00:00:00|    aqua|   33.2|554748717|
|2019-10-01 00:00:01|    NULL|  543.1|519107250|
|2019-10-01 00:00:01|  lenovo| 251.74|550050854|
|2019-10-01 00:00:04|   apple|1081.98|535871217|
+-------------------+--------+-------+---------+
only showing top 5 rows


None

[StructField('event_time', TimestampType(), True),
 StructField('brand', StringType(), True),
 StructField('price', DoubleType(), True),
 StructField('user_id', IntegerType(), True)]

In [4]:
# 4. Export: Write to processed data folder
output_path = "data/processed/prototype_output"
transformed_df.write.mode("overwrite").csv(output_path, header=True)

print(f"Data written to {output_path}")

Data written to data/processed/prototype_output
